# Module D: Transfer + Final Comparison (D2-D5)

**Zero-shot transfer, cross-task heatmap, and the controlled 4-way comparison table.**

| # | Task | Detail |
|---|------|--------|
| D2 | Zero-shot transfer Lift→Can / Lift→Square | Run Lift-trained predictor on Can/Square val triplets; CEM success |
| D3 | Cross-task transfer heatmap | 3×3: train on row, eval on column (Lift, Can, Square) |
| D4 | 4-way comparison table | Flat CEM, Hier CEM, Pure BC, JEPA-Aug BC on all 3 tasks |
| D5 | VC-1 baseline lookup | Published frozen-DINOv2 BC numbers for comparison |

**Prerequisites**:
- Lift: Modules A, B, C complete (`outputs/module_a_*`, `outputs/module_b_results.pt`, `outputs/module_c_*`)
- Can/Square: `Can_Square_Hier_experiment.ipynb` executed (S1-S16) for both tasks
- Can/Square BC: `module_c_bc_can_square.ipynb` executed for both tasks
- All embeddings cached in `data/embeddings/`

**Metrics design** (per project decision):
- **Primary 4-way table**: all 4 methods evaluated on goal-reaching metrics (Goal Cos, Success@0.90).
  BC methods are rolled out through the JEPA predictor to get comparable goal-reaching numbers.
- **Supplementary table**: Action MSE for BC methods (Pure BC, JEPA-Aug BC) vs VC-1 baseline.


## Section 1: Setup & Imports

In [ ]:
import sys, subprocess

def pip_install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

for pkg in ['torch>=2.0', 'matplotlib', 'scikit-learn', 'tqdm', 'numpy', 'pandas', 'scipy', 'pyarrow']:
    pip_install(pkg)

import os, math, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from tqdm.auto import tqdm

# -- Paths (Colab-aware) --
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/jepa_action")
    print("Mounted Google Drive. Make sure data/ is at MyDrive/jepa_action/data/")
except ImportError:
    ROOT = Path(os.getcwd()).resolve()
    if not (ROOT / "data" / "embeddings").exists():
        if (ROOT.parent / "data" / "embeddings").exists():
            ROOT = ROOT.parent
DATA_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / "outputs"
EMBED_DIR = DATA_DIR / "embeddings"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TASKS = ['lift', 'can', 'square']

# Config
class Config:
    embed_dim = 384
    action_dim = 7
    d_model = 256
    n_heads = 4
    n_layers_tr = 4
    n_layers_sg = 2
    batch_size = 64
    lr = 1e-4
    weight_decay = 0.05
    warmup_steps = 200
    use_amp = True
    dropout_p = 0.2
    seed = 42
    K_values = [1, 5, 10, 20]
    best_T = 8
    n_train_episodes = 160
    # CEM
    cem_n_pop = 128
    cem_elite_frac = 0.1
    cem_n_iters = 5
    success_threshold = 0.90
    # BC goal-reaching rollout
    bc_rollout_H = 20  # horizon for BC goal-reaching eval (steps)

cfg = Config()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
if device.type == 'cuda':
    torch.cuda.manual_seed_all(cfg.seed)
    torch.backends.cudnn.benchmark = True

print('Setup complete.')

## Section 2: Shared Model Definitions & Utilities

Re-define the TransformerPredictor, CEM planner, and helper functions used across modules.

In [ ]:
class ActionMLP(nn.Module):
    def __init__(self, action_dim=7, hidden_dim=128, out_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(action_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, out_dim),
        )
    def forward(self, a):
        return self.net(a)


class TransformerPredictor(nn.Module):
    def __init__(self, embed_dim=384, action_dim=7, d_model=256,
                 n_layers=4, n_heads=4, max_seq_len=64, dropout=0.2):
        super().__init__()
        self.d_model = d_model
        self.embed_proj = nn.Linear(embed_dim, d_model)
        self.action_mlp = ActionMLP(action_dim, hidden_dim=128, out_dim=d_model)
        self.pos_embed = nn.Parameter(torch.randn(1, max_seq_len, d_model) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, batch_first=True,
            dropout=dropout, activation='gelu')
        self.transformer = nn.TransformerEncoder(encoder_layer, n_layers)
        self.predictor = nn.Linear(d_model, embed_dim)
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1: nn.init.xavier_uniform_(p)

    def forward(self, embeds, actions):
        if embeds.dim() == 2:
            embeds = embeds.unsqueeze(1); actions = actions.unsqueeze(1)
        B, T, _ = embeds.shape
        x = self.embed_proj(embeds) + self.action_mlp(actions)
        x = x + self.pos_embed[:, :T, :]
        causal_mask = torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1)
        x = self.transformer(x, mask=causal_mask)
        return self.predictor(x[:, -1, :])


class BCPolicy(nn.Module):
    """Simple MLP policy: embedding (384) -> action (7)."""
    def __init__(self, embed_dim=384, action_dim=7, hidden_dims=[256, 128]):
        super().__init__()
        layers = []
        in_dim = embed_dim
        for h in hidden_dims:
            layers.extend([nn.Linear(in_dim, h), nn.ReLU()])
            in_dim = h
        layers.append(nn.Linear(in_dim, action_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, z):
        return self.net(z)


class ContextTripletDataset(Dataset):
    def __init__(self, emb, act, trip, T=1):
        self.embeddings = emb
        self.actions = act
        self.triplets = trip
        self.T = T
        valid = self.triplets[:, 0] >= (T - 1)
        self.triplets = self.triplets[valid]

    def __len__(self): return len(self.triplets)

    def __getitem__(self, idx):
        t_idx, a_idx, tpK_idx = self.triplets[idx]
        z_seq = self.embeddings[t_idx - self.T + 1 : t_idx + 1]
        a_seq = self.actions[t_idx - self.T + 1 : t_idx + 1]
        target = self.embeddings[tpK_idx]
        if z_seq.shape[0] < self.T:
            pad = self.T - z_seq.shape[0]
            z_seq = torch.cat([torch.zeros(pad, self.embeddings.shape[1]), z_seq], dim=0)
            a_seq = torch.cat([torch.zeros(pad, self.actions.shape[1]), a_seq], dim=0)
        return z_seq, a_seq, target


@torch.no_grad()
def compute_val_cosine(model, loader):
    model.eval()
    cs_list = []
    for z_seq, a_seq, target in loader:
        z_seq, a_seq, target = z_seq.to(device), a_seq.to(device), target.to(device)
        if cfg.use_amp and device.type == 'cuda':
            with autocast(): pred = model(z_seq, a_seq)
        else:
            pred = model(z_seq, a_seq)
        cs_list.append(F.cosine_similarity(pred, target, dim=-1).mean().item())
    return np.mean(cs_list)


@torch.no_grad()
def cem_plan(z_start, z_target, predictor, horizon, K, T,
             n_pop=128, elite_frac=0.1, n_iters=5, noise_scale=0.5):
    """Flat CEM planner (from module_b)."""
    n_elite = max(1, int(n_pop * elite_frac))
    mu = torch.zeros(horizon, cfg.action_dim, device=device)
    sigma = torch.ones(horizon, cfg.action_dim, device=device) * noise_scale
    best_seq, best_score = None, -float('inf')

    for it in range(n_iters):
        eps = torch.randn(n_pop, horizon, cfg.action_dim, device=device)
        actions = mu.unsqueeze(0) + sigma.unsqueeze(0) * eps
        scores = torch.zeros(n_pop, device=device)

        for i in range(n_pop):
            z_ctx = z_start.clone().unsqueeze(0).unsqueeze(0).repeat(1, T, 1)
            a_ctx = torch.zeros(1, T, cfg.action_dim, device=device)
            a_ctx[:, -1, :] = actions[i, 0]
            for step in range(horizon):
                a_cur = actions[i, step:step+1].unsqueeze(0)
                a_ctx = torch.cat([a_ctx[:, 1:, :], a_cur], dim=1)
                pred = predictor(z_ctx, a_ctx)
                z_ctx = torch.cat([z_ctx[:, 1:, :], pred.unsqueeze(1)], dim=1)
            z_pred = z_ctx[:, -1, :].squeeze(0)
            scores[i] = F.cosine_similarity(z_pred.unsqueeze(0), z_target.unsqueeze(0), dim=-1)

        elite_idx = torch.topk(scores, n_elite).indices
        ea = actions[elite_idx]
        mu = ea.mean(dim=0)
        sigma = ea.std(dim=0) + 1e-6
        if scores[elite_idx[0]].item() > best_score:
            best_score = scores[elite_idx[0]].item()
            best_seq = ea[0].cpu().numpy()
    return best_seq, best_score


@torch.no_grad()
def bc_goal_reaching_rollout(bc_policy, predictor, z_start, z_goal, H, T):
    """Roll out a BC policy through the JEPA predictor to measure goal reaching.

    At each step: a_t = BC(z_t), z_{t+1} = predictor(z_t, a_t).
    Returns (final_goal_cos, success_flag).
    """
    bc_policy.eval(); predictor.eval()
    z_cur = z_start.clone().to(device)
    z_ctx = z_cur.unsqueeze(0).unsqueeze(0).repeat(1, T, 1)
    a_ctx = torch.zeros(1, T, cfg.action_dim, device=device)

    for step in range(H):
        a_pred = bc_policy(z_cur.unsqueeze(0))  # (1, 7)
        a_cur = a_pred.unsqueeze(0)  # (1, 1, 7)
        a_ctx = torch.cat([a_ctx[:, 1:, :], a_cur], dim=1)
        if cfg.use_amp and device.type == 'cuda':
            with autocast(): pred = predictor(z_ctx, a_ctx)
        else:
            pred = predictor(z_ctx, a_ctx)
        z_ctx = torch.cat([z_ctx[:, 1:, :], pred.unsqueeze(1)], dim=1)
        z_cur = pred.squeeze(0)

    goal_cos = F.cosine_similarity(z_cur.unsqueeze(0), z_goal.unsqueeze(0).to(device), dim=-1).item()
    success = 1.0 if goal_cos >= cfg.success_threshold else 0.0
    return goal_cos, success


def cosine_loss(pred, target):
    return 1.0 - F.cosine_similarity(pred, target, dim=-1).mean()

def get_lr(step, warmup_steps, total_steps, base_lr):
    if step < warmup_steps:
        return base_lr * step / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return base_lr * 0.5 * (1.0 + math.cos(math.pi * progress))

print('Models and utilities ready.')

## Section 3: Data Loading Helpers

Load embeddings, actions, triplets, and per-episode parquet data for each task.
Lift uses the original (un-suffixed) cache files; Can/Square use `_{task}` suffixes.

In [ ]:
def load_task_data(task):
    """Load embeddings, actions, triplets, and parquet episode boundaries for a task."""
    if task == 'lift':
        emb_tr = torch.load(EMBED_DIR / 'embeddings_train.pt').float()
        emb_va = torch.load(EMBED_DIR / 'embeddings_val.pt').float()
        act_tr = torch.load(EMBED_DIR / 'actions_train.pt').float()
        act_va = torch.load(EMBED_DIR / 'actions_val.pt').float()
        parquet_dir = DATA_DIR / 'robomimic-ph-lift-image' / 'data' / 'chunk-000'
    else:
        emb_tr = torch.load(EMBED_DIR / f'embeddings_{task}_train.pt').float()
        emb_va = torch.load(EMBED_DIR / f'embeddings_{task}_val.pt').float()
        act_tr = torch.load(EMBED_DIR / f'actions_{task}_train.pt').float()
        act_va = torch.load(EMBED_DIR / f'actions_{task}_val.pt').float()
        parquet_dir = DATA_DIR / f'robomimic-ph-{task}-image' / 'data' / 'chunk-000'

    # Load parquet episode boundaries
    parquet_files = sorted(parquet_dir.glob('episode_*.parquet'))
    all_ep_actions = []
    all_ep_n_frames = []
    for pf in parquet_files:
        df = pd.read_parquet(pf)
        actions_raw = np.stack(df['action'].values).astype(np.float32)
        all_ep_actions.append(torch.tensor(actions_raw[::2]))
        all_ep_n_frames.append(len(actions_raw[::2]))

    # Build episode -> embedding index mapping
    train_offset = 0; val_offset = 0
    ep_to_emb_start = {}
    for ep_idx in range(len(parquet_files)):
        n_frames = all_ep_n_frames[ep_idx]
        if ep_idx < cfg.n_train_episodes:
            ep_to_emb_start[ep_idx] = ('train', train_offset)
            train_offset += n_frames
        else:
            ep_to_emb_start[ep_idx] = ('val', val_offset)
            val_offset += n_frames

    # Load triplets
    triplets = {}
    for K in cfg.K_values:
        if task == 'lift':
            tp = torch.load(EMBED_DIR / f'triplets_train_K{K}.pt')
            tv = torch.load(EMBED_DIR / f'triplets_val_K{K}.pt')
        else:
            tp = torch.load(EMBED_DIR / f'triplets_{task}_train_K{K}.pt')
            tv = torch.load(EMBED_DIR / f'triplets_{task}_val_K{K}.pt')
        triplets[K] = {'train': tp, 'val': tv}

    # Build val eval samples (z_start, z_goal per val episode)
    eval_samples = []
    for ep in range(cfg.n_train_episodes, len(parquet_files)):
        n_frames = all_ep_n_frames[ep]
        if n_frames < 3: continue
        split, start = ep_to_emb_start[ep]
        emb = emb_va if split == 'val' else emb_tr
        z_s = emb[start]
        z_g = emb[start + n_frames - 1]
        eval_samples.append({'ep': ep, 'z_start': z_s, 'z_goal': z_g, 'n_frames': n_frames})

    return {
        'task': task,
        'train_emb': emb_tr, 'val_emb': emb_va,
        'train_act': act_tr, 'val_act': act_va,
        'parquet_files': parquet_files,
        'all_ep_actions': all_ep_actions, 'all_ep_n_frames': all_ep_n_frames,
        'ep_to_emb_start': ep_to_emb_start,
        'triplets': triplets,
        'eval_samples': eval_samples,
    }


# Load all tasks
task_data = {}
for task in TASKS:
    try:
        task_data[task] = load_task_data(task)
        d = task_data[task]
        print(f'{task.upper()}: train={d["train_emb"].shape}, val={d["val_emb"].shape}, '
              f'eval_samples={len(d["eval_samples"])}, triplets K={list(d["triplets"].keys())}')
    except Exception as e:
        print(f'{task.upper()}: FAILED to load - {e}')
        print(f'  (Run Can_Square_Hier_experiment.ipynb for this task first)')

print('\nData loading complete.')

## Section 4: Load All Predictor Checkpoints

Load the trained TransformerPredictor for each task. Lift uses `module_a_transformer_K{K}_T8.pt`;
Can/Square use `transformer_{TASK}_K{K}_T8.pt` (from `Can_Square_Hier_experiment.ipynb` S5).

In [ ]:
def load_predictor(task, K, T=cfg.best_T):
    """Load a trained JEPA predictor for a task."""
    if task == 'lift':
        ckpt_path = OUTPUT_DIR / f'module_a_transformer_K{K}_T{T}.pt'
    else:
        ckpt_path = OUTPUT_DIR / f'transformer_{task}_K{K}_T{T}.pt'
    if not ckpt_path.exists():
        return None
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model = TransformerPredictor(
        embed_dim=cfg.embed_dim, action_dim=cfg.action_dim, d_model=cfg.d_model,
        n_layers=cfg.n_layers_tr, n_heads=cfg.n_heads, dropout=cfg.dropout_p,
    ).to(device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    return model


# Load K=1 predictors for all tasks (used for D3 heatmap + D4 BC rollout)
predictors = {}
for task in TASKS:
    if task not in task_data:
        continue
    model = load_predictor(task, K=1)
    if model is not None:
        predictors[task] = model
        print(f'Loaded {task.upper()} predictor K=1 T=8')
    else:
        print(f'{task.upper()}: predictor checkpoint missing')

# Also load Lift predictors for all K (for D2 zero-shot)
lift_predictors_allK = {}
for K in cfg.K_values:
    model = load_predictor('lift', K=K)
    if model is not None:
        lift_predictors_allK[K] = model
        print(f'Loaded Lift predictor K={K} T=8')

print('\nPredictor loading complete.')

## Section 5: D2 — Zero-Shot Transfer (Lift → Can / Lift → Square)

Take the Lift-trained predictor and evaluate it on Can and Square validation triplets **without fine-tuning**.
Also run flat CEM on Can/Square val episodes using the Lift predictor.

**Produces**:
- `outputs/d2_zero_shot.pt` — cosine sim + CEM success for each (train=Lift, eval=Can/Square) pair
- `outputs/d2_zero_shot.png` — bar chart comparing in-domain vs zero-shot cosine sim

In [ ]:
d2_results = {}

# In-domain baselines (Lift->Lift) for comparison
print('D2: Zero-shot transfer evaluation')
print('=' * 60)

for eval_task in TASKS:
    if eval_task not in task_data:
        print(f'\nSkipping {eval_task.upper()} (data not available)')
        continue
    d = task_data[eval_task]
    d2_results[eval_task] = {}

    for K in cfg.K_values:
        if K not in lift_predictors_allK:
            continue
        predictor = lift_predictors_allK[K]

        # Build val loader for eval_task at this K
        val_ds = ContextTripletDataset(d['val_emb'], d['val_act'], d['triplets'][K]['val'], T=cfg.best_T)
        val_ldr = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False,
                             pin_memory=(device.type=='cuda'))

        # Predictor cosine similarity (zero-shot if eval_task != lift)
        val_cos = compute_val_cosine(predictor, val_ldr)
        d2_results[eval_task][K] = {'val_cos': val_cos}
        tag = 'in-domain' if eval_task == 'lift' else 'zero-shot'
        print(f'  Lift->{eval_task.upper()} K={K:2d} ({tag}): val_cos={val_cos:.4f}')

print('\nD2 predictor cosine similarity done.')

In [ ]:
# CEM zero-shot evaluation: Lift predictor on Can/Square val episodes
for eval_task in ['can', 'square']:
    if eval_task not in task_data:
        continue
    d = task_data[eval_task]
    print(f'\nCEM zero-shot: Lift predictor -> {eval_task.upper()}')

    for K in [1, 5]:  # focus on short horizons for CEM
        if K not in lift_predictors_allK:
            continue
        predictor = lift_predictors_allK[K]
        horizon = max(1, 20 // K)

        goal_cos_list, success_list = [], []
        for sample in tqdm(d['eval_samples'], desc=f'Lift->{eval_task} K={K} CEM'):
            z_s = sample['z_start'].to(device)
            z_g = sample['z_goal'].to(device)
            _, score = cem_plan(z_s, z_g, predictor, horizon, K, cfg.best_T,
                                cfg.cem_n_pop, cfg.cem_elite_frac, cfg.cem_n_iters)
            goal_cos_list.append(score)
            success_list.append(1.0 if score >= cfg.success_threshold else 0.0)

        d2_results[eval_task][K]['cem_goal_cos'] = np.mean(goal_cos_list)
        d2_results[eval_task][K]['cem_success_rate'] = np.mean(success_list)
        print(f'  K={K}: CEM goal_cos={np.mean(goal_cos_list):.4f}, success={np.mean(success_list):.3f}')

# Save
torch.save(d2_results, OUTPUT_DIR / 'd2_zero_shot.pt')
print('\nSaved outputs/d2_zero_shot.pt')

In [ ]:
# Plot D2: bar chart of val_cos for Lift->Lift, Lift->Can, Lift->Square across K values
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(cfg.K_values))
width = 0.25
colors = {'lift': '#2ca02c', 'can': '#1f77b4', 'square': '#d62728'}

for ti, task in enumerate(TASKS):
    if task not in d2_results:
        continue
    cos_vals = [d2_results[task].get(K, {}).get('val_cos', 0) for K in cfg.K_values]
    label = f'Lift->{task.upper()}' + (' (in-domain)' if task == 'lift' else ' (zero-shot)')
    ax.bar(x + ti * width, cos_vals, width, label=label, color=colors[task], alpha=0.85)

ax.set_xlabel('Prediction Horizon K')
ax.set_ylabel('Validation Cosine Similarity')
ax.set_title('D2: Zero-Shot Transfer (Lift-trained predictor)')
ax.set_xticks(x + width)
ax.set_xticklabels([f'K={K}' for K in cfg.K_values])
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'd2_zero_shot.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary table
print(f'\n{"Train->Eval":<18}', end='')
for K in cfg.K_values:
    print(f'{"K="+str(K):<12}', end='')
print()
print('-' * (18 + 12 * len(cfg.K_values)))
for task in TASKS:
    if task not in d2_results:
        continue
    label = f'Lift->{task.upper()}'
    print(f'{label:<18}', end='')
    for K in cfg.K_values:
        v = d2_results[task].get(K, {}).get('val_cos', None)
        print(f'{v:<12.4f}' if v is not None else f'{"N/A":<12}', end='')
    print()

## Section 6: D3 — Cross-Task Transfer Heatmap (3×3)

Evaluate each task's K=1 predictor on every other task's val triplets.
Produces a 3×3 heatmap: rows = train task, columns = eval task, cell = val cosine similarity.

**Produces**:
- `outputs/d3_transfer_heatmap.pt` — 3×3 matrix of cosine similarities
- `outputs/d3_transfer_heatmap.png` — annotated heatmap figure

In [ ]:
# Build 3x3 transfer matrix (K=1)
transfer_matrix = np.full((3, 3), np.nan)
transfer_labels = []

for ti, train_task in enumerate(TASKS):
    if train_task not in predictors:
        print(f'{train_task.upper()} predictor missing, skipping row')
        continue
    predictor = predictors[train_task]

    for ei, eval_task in enumerate(TASKS):
        if eval_task not in task_data:
            continue
        d = task_data[eval_task]
        val_ds = ContextTripletDataset(d['val_emb'], d['val_act'], d['triplets'][1]['val'], T=cfg.best_T)
        val_ldr = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False,
                             pin_memory=(device.type=='cuda'))
        val_cos = compute_val_cosine(predictor, val_ldr)
        transfer_matrix[ti, ei] = val_cos
        print(f'  {train_task.upper()}->{eval_task.upper()}: val_cos={val_cos:.4f}')

# Save
torch.save({'matrix': transfer_matrix, 'tasks': TASKS}, OUTPUT_DIR / 'd3_transfer_heatmap.pt')
print('\nSaved outputs/d3_transfer_heatmap.pt')

In [ ]:
# Plot 3x3 heatmap
fig, ax = plt.subplots(figsize=(7, 6))
# Mask NaN cells (if Can/Square not available)
masked = np.ma.array(transfer_matrix, mask=np.isnan(transfer_matrix))
cmap = LinearSegmentedColormap.from_list('cos', ['#d62728', '#ff7f0e', '#2ca02c'])
cmap.set_bad(color='lightgray')
im = ax.imshow(masked, cmap=cmap, vmin=0.85, vmax=1.0, aspect='auto')

# Annotate cells
for i in range(3):
    for j in range(3):
        v = transfer_matrix[i, j]
        if not np.isnan(v):
            color = 'white' if v < 0.92 else 'black'
            ax.text(j, i, f'{v:.4f}', ha='center', va='center', fontsize=14, color=color, fontweight='bold')
        else:
            ax.text(j, i, 'N/A', ha='center', va='center', fontsize=12, color='gray')

ax.set_xticks(range(3)); ax.set_xticklabels([t.upper() for t in TASKS], fontsize=12)
ax.set_yticks(range(3)); ax.set_yticklabels([t.upper() for t in TASKS], fontsize=12)
ax.set_xlabel('Eval Task', fontsize=13)
ax.set_ylabel('Train Task', fontsize=13)
ax.set_title('D3: Cross-Task Transfer Heatmap (K=1, val cosine sim)', fontsize=13)
plt.colorbar(im, ax=ax, label='Cosine Similarity', shrink=0.8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'd3_transfer_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Print matrix
print(f'\n{"":<12}', end='')
for t in TASKS: print(f'{t.upper():<12}', end='')
print()
print('-' * 48)
for i, train_task in enumerate(TASKS):
    print(f'{train_task.upper():<12}', end='')
    for j in range(3):
        v = transfer_matrix[i, j]
        print(f'{v:<12.4f}' if not np.isnan(v) else f'{"N/A":<12}', end='')
    print()

## Section 7: D4 — Compile CEM Results (Flat + Hierarchical)

Load existing CEM evaluation results from Module B (Lift) and `Can_Square_Hier_experiment.ipynb` (Can/Square).
Select the best config per task (highest goal_cos among flat configs, highest among hier configs).

In [ ]:
# Load Lift CEM results (from module_b)
d4_cem = {}  # {task: {'flat': {goal_cos, success_rate, config}, 'hier': {...}}}

if (OUTPUT_DIR / 'module_b_results.pt').exists():
    lift_b = torch.load(OUTPUT_DIR / 'module_b_results.pt', weights_only=False)
    # Best flat CEM config for Lift
    flat_results = lift_b.get('flat_cem_results', {})
    if flat_results:
        best_flat_key = max(flat_results.keys(), key=lambda k: flat_results[k].get('goal_cos', 0))
        d4_cem.setdefault('lift', {})['flat'] = {
            'goal_cos': flat_results[best_flat_key]['goal_cos'],
            'success_rate': flat_results[best_flat_key]['success_rate'],
            'config': best_flat_key,
        }
        print(f'Lift Flat CEM best: {best_flat_key} -> goal_cos={flat_results[best_flat_key]["goal_cos"]:.4f}, '
              f'success={flat_results[best_flat_key]["success_rate"]:.3f}')
    # Best hier CEM config for Lift
    hier_results = lift_b.get('hier_cem_results', {})
    if hier_results:
        best_hier_key = max(hier_results.keys(), key=lambda k: hier_results[k].get('goal_cos', 0))
        d4_cem.setdefault('lift', {})['hier'] = {
            'goal_cos': hier_results[best_hier_key]['goal_cos'],
            'success_rate': hier_results[best_hier_key]['success_rate'],
            'config': best_hier_key,
        }
        print(f'Lift Hier CEM best: {best_hier_key} -> goal_cos={hier_results[best_hier_key]["goal_cos"]:.4f}, '
              f'success={hier_results[best_hier_key]["success_rate"]:.3f}')
else:
    print('WARNING: module_b_results.pt not found')

# Load Can/Square CEM results
for task in ['can', 'square']:
    path = OUTPUT_DIR / f'can_square_hier_{task}_results.pt'
    if not path.exists():
        print(f'WARNING: {path.name} not found (run Can_Square_Hier_experiment.ipynb for {task})')
        continue
    r = torch.load(path, weights_only=False)
    flat_results = r.get('flat_cem_results', {})
    hier_results = r.get('hier_cem_results', {})
    if flat_results:
        best_flat_key = max(flat_results.keys(), key=lambda k: flat_results[k].get('goal_cos', 0))
        d4_cem.setdefault(task, {})['flat'] = {
            'goal_cos': flat_results[best_flat_key]['goal_cos'],
            'success_rate': flat_results[best_flat_key]['success_rate'],
            'config': best_flat_key,
        }
        print(f'{task.upper()} Flat CEM best: {best_flat_key} -> goal_cos={flat_results[best_flat_key]["goal_cos"]:.4f}')
    if hier_results:
        best_hier_key = max(hier_results.keys(), key=lambda k: hier_results[k].get('goal_cos', 0))
        d4_cem.setdefault(task, {})['hier'] = {
            'goal_cos': hier_results[best_hier_key]['goal_cos'],
            'success_rate': hier_results[best_hier_key]['success_rate'],
            'config': best_hier_key,
        }
        print(f'{task.upper()} Hier CEM best: {best_hier_key} -> goal_cos={hier_results[best_hier_key]["goal_cos"]:.4f}')

print('\nD4 CEM results compiled.')

## Section 8: D4 — BC Goal-Reaching Evaluation

Roll out BC policies (pure + JEPA-augmented) through the JEPA predictor to get goal-reaching metrics
(goal_cos, success_rate) comparable with CEM methods.

Also collect Action MSE for the supplementary table.

**BC checkpoints**:
- Lift: `outputs/module_c_bc_baseline.pt`, `outputs/module_c_bc_aug_alpha{best}.pt`
- Can/Square: `outputs/module_c_bc_baseline_{TASK}.pt`, `outputs/module_c_bc_aug_{TASK}_alpha{best}.pt`

In [ ]:
def load_bc_policy(task, augmented=False, alpha=None):
    """Load a BC policy checkpoint."""
    if task == 'lift':
        if augmented:
            if alpha is None:
                # Find best alpha (lowest val_mse)
                best_mse = float('inf'); best_path = None
                for a in cfg.alpha_values if hasattr(cfg, 'alpha_values') else [0.0, 0.25, 0.5, 0.75, 1.0]:
                    p = OUTPUT_DIR / f'module_c_bc_aug_alpha{a}.pt'
                    if p.exists():
                        ckpt = torch.load(p, map_location='cpu', weights_only=False)
                        if ckpt['val_mse'] < best_mse:
                            best_mse = ckpt['val_mse']; best_path = p; best_alpha = a
                if best_path is None:
                    return None, None, None
                ckpt = torch.load(best_path, map_location=device, weights_only=False)
                return _build_bc(ckpt), ckpt['val_mse'], best_alpha
            else:
                p = OUTPUT_DIR / f'module_c_bc_aug_alpha{alpha}.pt'
        else:
            p = OUTPUT_DIR / 'module_c_bc_baseline.pt'
    else:
        if augmented:
            if alpha is None:
                best_mse = float('inf'); best_path = None; best_alpha = None
                for a in [0.0, 0.25, 0.5, 0.75, 1.0]:
                    p = OUTPUT_DIR / f'module_c_bc_aug_{task}_alpha{a}.pt'
                    if p.exists():
                        ckpt = torch.load(p, map_location='cpu', weights_only=False)
                        if ckpt['val_mse'] < best_mse:
                            best_mse = ckpt['val_mse']; best_path = p; best_alpha = a
                if best_path is None:
                    return None, None, None
                ckpt = torch.load(best_path, map_location=device, weights_only=False)
                return _build_bc(ckpt), ckpt['val_mse'], best_alpha
            else:
                p = OUTPUT_DIR / f'module_c_bc_aug_{task}_alpha{alpha}.pt'
        else:
            p = OUTPUT_DIR / f'module_c_bc_baseline_{task}.pt'

    if not p.exists():
        return None, None, None
    ckpt = torch.load(p, map_location=device, weights_only=False)
    return _build_bc(ckpt), ckpt.get('val_mse'), ckpt.get('alpha', None)


def _build_bc(ckpt):
    model = BCPolicy(embed_dim=cfg.embed_dim, action_dim=cfg.action_dim,
                     hidden_dims=[256, 128]).to(device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    return model


print('BC loading helpers ready.')

In [ ]:
# Evaluate BC goal-reaching for all tasks
d4_bc = {}  # {task: {'pure': {goal_cos, success_rate, action_mse}, 'aug': {...}}}

for task in TASKS:
    if task not in task_data or task not in predictors:
        print(f'\nSkipping {task.upper()} (data or predictor missing)')
        continue
    d = task_data[task]
    predictor = predictors[task]
    eval_samples = d['eval_samples']
    print(f'\nEvaluating BC goal-reaching for {task.upper()} ({len(eval_samples)} samples)...')

    for bc_type in ['pure', 'aug']:
        augmented = (bc_type == 'aug')
        bc_model, val_mse, alpha = load_bc_policy(task, augmented=augmented)
        if bc_model is None:
            print(f'  {bc_type}: checkpoint not found')
            continue

        goal_cos_list, success_list = [], []
        for sample in tqdm(eval_samples, desc=f'{task.upper()} {bc_type} BC rollout'):
            z_s = sample['z_start'].to(device)
            z_g = sample['z_goal'].to(device)
            gc, succ = bc_goal_reaching_rollout(bc_model, predictor, z_s, z_g,
                                                H=cfg.bc_rollout_H, T=cfg.best_T)
            goal_cos_list.append(gc)
            success_list.append(succ)

        d4_bc.setdefault(task, {})[bc_type] = {
            'goal_cos': np.mean(goal_cos_list),
            'success_rate': np.mean(success_list),
            'action_mse': val_mse,
            'alpha': alpha,
        }
        print(f'  {bc_type} BC (alpha={alpha}): goal_cos={np.mean(goal_cos_list):.4f}, '
              f'success={np.mean(success_list):.3f}, action_mse={val_mse:.6f}')

# Save
torch.save(d4_bc, OUTPUT_DIR / 'd4_bc_goal_reaching.pt')
print('\nSaved outputs/d4_bc_goal_reaching.pt')

## Section 9: D5 — VC-1 Baseline Lookup

Look up published VC-1 (and R3M, MVP) frozen-DINOv2 BC numbers for Robomimic Lift/Can/Square.
These are **manually entered** from the literature — see the VC-1 paper (Nair et al. 2023,
"VC-1: A Universal Visual Controller") and related work.

**To fill in**: Find the exact BC success rates / action MSE for frozen DINOv2 ViT-S on
Robomimic Lift, Can, Square. The numbers below are placeholders — replace with actual values
from the paper. (This is task K4 for Kshitiz.)

In [ ]:
# VC-1 baseline numbers — MANUALLY ENTERED from literature
# Sources: VC-1 paper (Nair et al. 2023), R3M (Nair et al. 2022), MVP (Xiao et al. 2022)
# Metric: BC success rate (%) on Robomimic proficient-human split with frozen visual encoder.
#
# TODO: Replace placeholder values with actual published numbers.
#       Key reference: VC-1 Table 2 (Robomimic results with frozen DINOv2).
#       If action MSE is not reported, compute from success rate using the
#       approximate relationship, or leave as N/A.

vc1_baselines = {
    'lift': {
        'success_rate': None,    # TODO: e.g. 0.92 (from VC-1 paper)
        'action_mse': None,      # TODO: if reported
        'goal_cos': None,        # N/A (VC-1 doesn't report latent goal cos)
        'source': 'VC-1 paper, Table X (TODO: fill citation)',
    },
    'can': {
        'success_rate': None,
        'action_mse': None,
        'goal_cos': None,
        'source': 'VC-1 paper, Table X (TODO: fill citation)',
    },
    'square': {
        'success_rate': None,
        'action_mse': None,
        'goal_cos': None,
        'source': 'VC-1 paper, Table X (TODO: fill citation)',
    },
}

print('VC-1 baselines (placeholder — fill from literature):')
for task, v in vc1_baselines.items():
    print(f'  {task.upper()}: success={v["success_rate"]}, action_mse={v["action_mse"]}')

print('\nNOTE: Replace None values with actual numbers from the VC-1 paper.')
print('      This is task K4 (Kshitiz) per WORK_SHEET.md.')

## Section 10: D4 — Compile 4-Way Comparison Tables

### Primary Table: Goal-Reaching Metrics (All 4 Methods + VC-1)

| Method | Lift (Goal Cos / Success) | Can | Square |
|--------|---------------------------|-----|--------|
| Flat CEM (best) | from D4 CEM | ... | ... |
| Hier CEM (best) | from D4 CEM | ... | ... |
| Pure BC (rolled out) | from D4 BC | ... | ... |
| JEPA-Aug BC (rolled out) | from D4 BC | ... | ... |
| VC-1 (frozen DINOv2 BC) | from D5 | ... | ... |

### Supplementary Table: Action MSE (BC Methods + VC-1)

| Method | Lift | Can | Square |
|--------|------|-----|--------|
| Pure BC | from D4 BC | ... | ... |
| JEPA-Aug BC (best α) | from D4 BC | ... | ... |
| VC-1 | from D5 | ... | ... |

In [ ]:
# ── Primary 4-way table: Goal Cos + Success Rate ──
methods = ['Flat CEM (best)', 'Hier CEM (best)', 'Pure BC', 'JEPA-Aug BC', 'VC-1']

print('=' * 90)
print('D4: Primary 4-Way Comparison — Goal-Reaching Metrics')
print('=' * 90)

header = f'{"Method":<22}'
for task in TASKS:
    header += f'{task.upper() + " (cos/succ)":<22}'
print(header)
print('-' * 88)

four_way_primary = {}
for method in methods:
    row = f'{method:<22}'
    four_way_primary[method] = {}
    for task in TASKS:
        vals = None
        if method == 'Flat CEM (best)':
            v = d4_cem.get(task, {}).get('flat', {})
            if v:
                vals = (v['goal_cos'], v['success_rate'])
        elif method == 'Hier CEM (best)':
            v = d4_cem.get(task, {}).get('hier', {})
            if v:
                vals = (v['goal_cos'], v['success_rate'])
        elif method == 'Pure BC':
            v = d4_bc.get(task, {}).get('pure', {})
            if v:
                vals = (v['goal_cos'], v['success_rate'])
        elif method == 'JEPA-Aug BC':
            v = d4_bc.get(task, {}).get('aug', {})
            if v:
                vals = (v['goal_cos'], v['success_rate'])
        elif method == 'VC-1':
            v = vc1_baselines.get(task, {})
            if v.get('goal_cos') is not None:
                vals = (v['goal_cos'], v.get('success_rate'))
            elif v.get('success_rate') is not None:
                vals = (None, v['success_rate'])

        if vals is not None:
            gc, sr = vals
            gc_str = f'{gc:.3f}' if gc is not None else 'N/A'
            sr_str = f'{sr:.3f}' if sr is not None else 'N/A'
            row += f'{gc_str + "/" + sr_str:<22}'
            four_way_primary[method][task] = {'goal_cos': gc, 'success_rate': sr}
        else:
            row += f'{"N/A":<22}'
    print(row)

print()

# ── Supplementary table: Action MSE ──
print('=' * 60)
print('D4: Supplementary — Action MSE (BC methods vs VC-1)')
print('=' * 60)

header = f'{"Method":<22}'
for task in TASKS:
    header += f'{task.upper():<14}'
print(header)
print('-' * 64)

four_way_supp = {}
for method in ['Pure BC', 'JEPA-Aug BC', 'VC-1']:
    row = f'{method:<22}'
    four_way_supp[method] = {}
    for task in TASKS:
        mse = None
        if method == 'Pure BC':
            v = d4_bc.get(task, {}).get('pure', {})
            mse = v.get('action_mse')
        elif method == 'JEPA-Aug BC':
            v = d4_bc.get(task, {}).get('aug', {})
            mse = v.get('action_mse')
        elif method == 'VC-1':
            v = vc1_baselines.get(task, {})
            mse = v.get('action_mse')

        if mse is not None:
            row += f'{mse:<14.6f}'
            four_way_supp[method][task] = mse
        else:
            row += f'{"N/A":<14}'
    print(row)

# Save both tables
torch.save({
    'primary': four_way_primary,
    'supplementary': four_way_supp,
    'd4_cem': d4_cem,
    'd4_bc': d4_bc,
    'vc1_baselines': vc1_baselines,
    'd2_results': d2_results,
    'd3_matrix': transfer_matrix,
}, OUTPUT_DIR / 'd4_four_way_table.pt')
print('\nSaved outputs/d4_four_way_table.pt')

## Section 11: D4 — Visualisation (Grouped Bar Chart)

Bar chart comparing the 4 methods (excluding VC-1 if not available) across the 3 tasks on goal cosine similarity.

In [ ]:
# Grouped bar chart: 4 methods x 3 tasks (goal_cos)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

plot_methods = ['Flat CEM (best)', 'Hier CEM (best)', 'Pure BC', 'JEPA-Aug BC']
plot_colors = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3']
available_tasks = [t for t in TASKS if t in d4_cem or t in d4_bc]
x = np.arange(len(available_tasks))
width = 0.2

# Goal cosine
ax = axes[0]
for mi, method in enumerate(plot_methods):
    vals = []
    for task in available_tasks:
        if method == 'Flat CEM (best)':
            v = d4_cem.get(task, {}).get('flat', {}).get('goal_cos')
        elif method == 'Hier CEM (best)':
            v = d4_cem.get(task, {}).get('hier', {}).get('goal_cos')
        elif method == 'Pure BC':
            v = d4_bc.get(task, {}).get('pure', {}).get('goal_cos')
        elif method == 'JEPA-Aug BC':
            v = d4_bc.get(task, {}).get('aug', {}).get('goal_cos')
        else:
            v = None
        vals.append(v if v is not None else 0)
    bars = ax.bar(x + mi * width, vals, width, label=method, color=plot_colors[mi], alpha=0.85)
    for bar, val in zip(bars, vals):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                    f'{val:.3f}', ha='center', va='bottom', fontsize=7, rotation=90)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels([t.upper() for t in available_tasks], fontsize=12)
ax.set_ylabel('Goal Cosine Similarity')
ax.set_title('D4: 4-Way Comparison — Goal Cosine Similarity')
ax.legend(fontsize=9, loc='lower right')
ax.grid(True, alpha=0.3, axis='y')

# Success rate
ax = axes[1]
for mi, method in enumerate(plot_methods):
    vals = []
    for task in available_tasks:
        if method == 'Flat CEM (best)':
            v = d4_cem.get(task, {}).get('flat', {}).get('success_rate')
        elif method == 'Hier CEM (best)':
            v = d4_cem.get(task, {}).get('hier', {}).get('success_rate')
        elif method == 'Pure BC':
            v = d4_bc.get(task, {}).get('pure', {}).get('success_rate')
        elif method == 'JEPA-Aug BC':
            v = d4_bc.get(task, {}).get('aug', {}).get('success_rate')
        else:
            v = None
        vals.append(v if v is not None else 0)
    bars = ax.bar(x + mi * width, vals, width, label=method, color=plot_colors[mi], alpha=0.85)
    for bar, val in zip(bars, vals):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                    f'{val:.2f}', ha='center', va='bottom', fontsize=7, rotation=90)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels([t.upper() for t in available_tasks], fontsize=12)
ax.set_ylabel(f'Success@{int(cfg.success_threshold * 100)}')
ax.set_title('D4: 4-Way Comparison — Success Rate')
ax.legend(fontsize=9, loc='lower right')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'd4_four_way_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 12: Export LaTeX Table

Generate a publication-ready LaTeX table for the paper (`outputs/d4_four_way_table.tex`).

In [ ]:
# Generate LaTeX table for the primary 4-way comparison
def make_latex_table():
    lines = []
    lines.append(r'\begin{table}[h]\centering\small\setlength{\tabcolsep}{4pt}')
    lines.append(r'\caption{4-way comparison: goal cosine similarity and Success@0.90 across Robomimic tasks.}')
    lines.append(r'\label{tab:four_way}')
    n_tasks = len(TASKS)
    col_spec = '@{}l' + 'c' * n_tasks + '@{}'
    lines.append(r'\begin{tabular}{' + col_spec + '}')
    lines.append(r'\toprule')
    # Header
    header = 'Method & ' + ' & '.join(t.capitalize() for t in TASKS) + ' \\\\'
    lines.append(header)
    lines.append(r'\midrule')
    # Rows
    for method in methods:
        cells = [method]
        for task in TASKS:
            v = four_way_primary.get(method, {}).get(task, {})
            gc = v.get('goal_cos')
            sr = v.get('success_rate')
            if gc is not None and sr is not None:
                cells.append(f'{gc:.3f} / {sr:.2f}')
            elif sr is not None:
                cells.append(f'--/ {sr:.2f}')
            elif gc is not None:
                cells.append(f'{gc:.3f} /--')
            else:
                cells.append('--')
        lines.append(' & '.join(cells) + ' \\\\')
    lines.append(r'\bottomrule')
    lines.append(r'\end{tabular}')
    lines.append(r'\end{table}')
    return '\n'.join(lines)


latex_src = make_latex_table()
tex_path = OUTPUT_DIR / 'd4_four_way_table.tex'
with open(tex_path, 'w') as f:
    f.write(latex_src)
print(f'Saved {tex_path}')
print()
print(latex_src)

## Section 13: Save All Module D Results & Summary

In [ ]:
# Aggregate all Module D results
module_d_results = {
    'd2_zero_shot': d2_results,
    'd3_transfer_matrix': transfer_matrix,
    'd4_cem': d4_cem,
    'd4_bc': d4_bc,
    'd4_primary_table': four_way_primary,
    'd4_supplementary_table': four_way_supp,
    'd5_vc1_baselines': vc1_baselines,
    'config': {k: v for k, v in vars(cfg).items() if not k.startswith('__')},
}

torch.save(module_d_results, OUTPUT_DIR / 'module_d_results.pt')
print('Saved outputs/module_d_results.pt')

# Summary
print()
print('=' * 70)
print('Module D Complete')
print('=' * 70)

print(f'\nD2 Zero-Shot Transfer (Lift predictor):')
for task in TASKS:
    if task in d2_results:
        for K in [1, 5]:
            v = d2_results[task].get(K, {})
            if v:
                tag = 'in-domain' if task == 'lift' else 'zero-shot'
                cem_str = f", CEM cos={v.get('cem_goal_cos', 'N/A'):.4f}" if 'cem_goal_cos' in v else ''
                print(f'  Lift->{task.upper()} K={K} ({tag}): val_cos={v["val_cos"]:.4f}{cem_str}')

print(f'\nD3 Cross-Task Transfer Heatmap (K=1):')
for i, train_task in enumerate(TASKS):
    for j, eval_task in enumerate(TASKS):
        v = transfer_matrix[i, j]
        if not np.isnan(v):
            print(f'  {train_task.upper()}->{eval_task.upper()}: {v:.4f}')

print(f'\nD4 4-Way Comparison (Goal Cos / Success):')
for method in methods:
    parts = []
    for task in TASKS:
        v = four_way_primary.get(method, {}).get(task, {})
        gc = v.get('goal_cos'); sr = v.get('success_rate')
        if gc is not None:
            parts.append(f'{task.upper()}={gc:.3f}/{sr:.2f}')
        elif sr is not None:
            parts.append(f'{task.upper()}=--/{sr:.2f}')
    if parts:
        print(f'  {method}: {", ".join(parts)}')

print(f'\nD5 VC-1 Baselines:')
for task, v in vc1_baselines.items():
    sr = v.get('success_rate')
    print(f'  {task.upper()}: success={sr if sr is not None else "TODO (fill from VC-1 paper)"}')

print(f'\nFigures saved:')
for f in sorted(OUTPUT_DIR.glob('d[2-4]_*.png')):
    print(f'  {f.name}')
print(f'\nLaTeX table: outputs/d4_four_way_table.tex')